# 🤖 PR Governance Agent — Live Demo

**No API keys required.** This notebook connects to the already-deployed agent and triggers a real risk analysis.

---
### How it works:
1. You push a branch to the demo GitHub repo (or use an existing PR)
2. This notebook calls the live agent directly
3. You see the real AI risk analysis — the same output that gets posted to GitHub

> 💡 **Nothing to install. Nothing to configure. Just run the cells.**

## Step 1: Configure the Demo Target
Just fill in the repo and PR number you want to analyze.

In [ ]:
# ── CONFIGURE YOUR DEMO ──────────────────────────────────────
REPO = "lmudu2/risk-analyzer-poc"   # GitHub repo (owner/repo)
PR_NUMBER = "1"                      # PR number to analyze (as string)
# ─────────────────────────────────────────────────────────────

# Live Agent Gateway URL (already deployed — no keys needed)
GATEWAY_URL = "https://zrfcfcrgrwublxw22es6gvuidq0ckmpm.lambda-url.us-east-1.on.aws/"

print(f"✅ Target: {REPO} — PR #{PR_NUMBER}")
print(f"✅ Agent Endpoint: {GATEWAY_URL}")

## Step 2: Trigger Risk Analysis
This sends the PR to the live agent for analysis. The same logic that runs automatically on every push.

In [ ]:
import urllib.request
import json

# Build the simulated GitHub webhook payload
payload = {
    "action": "opened",
    "pull_request": {
        "number": int(PR_NUMBER),
        "title": f"Demo Analysis for PR #{PR_NUMBER}",
        "user": {"login": "demo-presenter", "type": "User"}
    },
    "repository": {
        "full_name": REPO
    },
    "installation": {"id": 12345678},
    "sender": {"login": "demo-presenter", "type": "User"}
}

print(f"⏳ Sending PR #{PR_NUMBER} to the agent for risk analysis...")
print("   This may take 5-10 seconds while the AI analyzes the code diff.\n")

try:
    req = urllib.request.Request(
        GATEWAY_URL,
        data=json.dumps(payload).encode('utf-8'),
        headers={
            'Content-Type': 'application/json',
            'X-GitHub-Event': 'pull_request'
        },
        method='POST'
    )
    with urllib.request.urlopen(req, timeout=30) as res:
        status = res.getcode()
        body = res.read().decode('utf-8')

    print(f"✅ Agent received the request (HTTP {status})")
    print("\n📨 Agent is now working in the background:")
    print("   → Fetching the PR code diff from GitHub")
    print("   → Running AI risk analysis (Llama 3 / Gemini)")
    print("   → If HIGH RISK: closing PR + creating Jira ticket + sending approval email")
    print("   → If LOW RISK: auto-merging PR")
    print(f"\n🔗 Watch it live: https://github.com/{REPO}/pulls")

except Exception as e:
    print(f"❌ Error: {e}")
    print("   Make sure the agent is deployed and the Gateway URL is reachable.")

## Step 3: Simulate a HIGH RISK Change
Trigger the agent with a payload that looks like a dangerous database schema change.

In [ ]:
# Simulate a HIGH RISK PR (schema change matching incident history)
high_risk_payload = {
    "action": "opened",
    "pull_request": {
        "number": 999,
        "title": "[DEMO] Refactor: drop user_id column from users table",
        "user": {"login": "demo-presenter", "type": "User"}
    },
    "repository": {"full_name": REPO},
    "installation": {"id": 12345678},
    "sender": {"login": "demo-presenter", "type": "User"}
}

print("⏳ Sending a HIGH RISK payload (schema change)...\n")

try:
    req = urllib.request.Request(
        GATEWAY_URL,
        data=json.dumps(high_risk_payload).encode('utf-8'),
        headers={'Content-Type': 'application/json', 'X-GitHub-Event': 'pull_request'},
        method='POST'
    )
    with urllib.request.urlopen(req, timeout=30) as res:
        print(f"✅ Agent received HIGH RISK payload (HTTP {res.getcode()})")
    print("\n🔴 The agent will now:")
    print("   → Classify this as HIGH RISK (schema change matches OUTAGE-2024-06)")
    print("   → Auto-close the PR")
    print("   → Create a Jira ticket for manager approval")
    print("   → Send an approval email")
    print(f"\n🔗 Watch it live: https://github.com/{REPO}/pulls")
except Exception as e:
    print(f"❌ Error: {e}")

## Step 4: Simulate a LOW RISK Change
Trigger the agent with a safe documentation update.

In [ ]:
# Simulate a LOW RISK PR (documentation update)
low_risk_payload = {
    "action": "opened",
    "pull_request": {
        "number": 998,
        "title": "[DEMO] docs: update README with usage instructions",
        "user": {"login": "demo-presenter", "type": "User"}
    },
    "repository": {"full_name": REPO},
    "installation": {"id": 12345678},
    "sender": {"login": "demo-presenter", "type": "User"}
}

print("⏳ Sending a LOW RISK payload (docs update)...\n")

try:
    req = urllib.request.Request(
        GATEWAY_URL,
        data=json.dumps(low_risk_payload).encode('utf-8'),
        headers={'Content-Type': 'application/json', 'X-GitHub-Event': 'pull_request'},
        method='POST'
    )
    with urllib.request.urlopen(req, timeout=30) as res:
        print(f"✅ Agent received LOW RISK payload (HTTP {res.getcode()})")
    print("\n🟢 The agent will now:")
    print("   → Classify this as LOW RISK (documentation only)")
    print("   → Auto-merge the PR immediately")
    print(f"\n🔗 Watch it live: https://github.com/{REPO}/pulls")
except Exception as e:
    print(f"❌ Error: {e}")